# 16. 强化学习基础与 MDP 体系
强化学习研究的是**序列决策问题**：智能体并不是一次性输出答案，而是在环境中持续观察、行动、接收反馈，并通过多步交互最大化长期累计回报。与监督学习直接拟合标注不同，强化学习的难点在于：

- 监督信号往往是**延迟到达**的，当前动作的质量要靠未来回报来判断。
- 数据分布由策略本身决定，策略一变，采样到的状态分布也会改变。
- 智能体必须同时处理**探索**与**利用**之间的张力。

## 强化学习与马尔可夫决策过程的形式化定义

            强化学习通常形式化为马尔可夫决策过程（Markov Decision Process, MDP）：
$$
\mathcal{M} = (\mathcal{S}, \mathcal{A}, P, R, \gamma)
$$
其中 $\mathcal{S}$ 为状态空间，$\mathcal{A}$ 为动作空间，$P(s' \mid s, a)$ 为状态转移概率，$R(s,a)$ 或 $R(s,a,s')$ 为奖励函数，$\gamma \in [0,1)$ 为折扣因子。策略 $\pi(a\mid s)$ 定义智能体在状态 $s$ 下采取动作 $a$ 的概率分布。强化学习的核心目标是求解最优策略
$$
\pi^\star = \arg\max_\pi \mathbb{E}_{\tau \sim \pi}\left[\sum_{t=0}^{T} \gamma^t r_t\right].
$$

## 输入、输出与参数化方式

            强化学习的输入不再是独立同分布样本，而是交互序列
$$
\tau = (s_0, a_0, r_0, s_1, a_1, r_1, \dots).
$$
输出可以是：

- 直接给出动作分布的策略 $\pi(a\mid s)$；
- 估计状态价值 $V^\pi(s)$；
- 估计动作价值 $Q^\pi(s,a)$；
- 同时学习策略与价值函数。

对于有限状态空间，策略和值函数可以用表格表示；对高维状态空间，则需要用神经网络进行函数逼近。

## 结构分解与信息流

            强化学习的结构由“策略 - 环境 - 反馈”闭环构成。一个标准决策周期包含四步：

1. 智能体接收当前状态 $s_t$；
2. 根据策略 $\pi(a\mid s_t)$ 选择动作 $a_t$；
3. 环境依据 $P(s_{t+1}\mid s_t, a_t)$ 发生转移并返回奖励 $r_t$；
4. 智能体用该反馈更新策略或价值估计。

这种闭环意味着强化学习中的数据生成过程与模型参数紧密耦合。策略改变之后，状态访问分布、奖励统计和训练难度都会随之改变。

## 优化目标与训练机制

            强化学习的统一优化目标是最大化期望累计回报。围绕这一目标形成了三条基本路线：

- 动态规划（DP）：假设环境模型已知，直接做 Bellman 备份；
- 蒙特卡洛（MC）：用完整轨迹回报估计价值，无偏但方差大；
- 时序差分（TD）：用一步 bootstrap 目标更新价值，有偏但方差更低。

价值函数定义为
$$
V^\pi(s) = \mathbb{E}_\pi\left[\sum_{t=0}^{\infty}\gamma^t r_t \mid s_0=s\right], \qquad
Q^\pi(s,a) = \mathbb{E}_\pi\left[\sum_{t=0}^{\infty}\gamma^t r_t \mid s_0=s, a_0=a\right].
$$
它们满足 Bellman 方程，是后续 DQN、Actor-Critic、PPO、SAC 等方法的共同基础。

## 计算复杂度、统计性质与工程代价

            强化学习的计算难点不只是单次前向计算，而是样本复杂度、探索成本和目标非平稳性。即使模型很小，只要奖励稀疏、状态空间大或信用分配链条长，训练就会显著变难。

从统计角度看，强化学习的方差来源包括：随机环境转移、随机策略、有限采样和 bootstrap 误差。从工程角度看，训练稳定性依赖于奖励尺度、折扣因子、探索策略、目标网络、经验回放、归一化和优势估计等细节。

## 与相邻模型的差异

            与监督学习相比，强化学习不依赖固定标签，而是从交互中构造训练信号。与 bandit 问题相比，MDP 额外引入了状态转移和长期信用分配。与搜索算法相比，强化学习不是在单次推理时规划一棵树，而是在参数空间中学习可重复执行的策略。

这一差异决定了强化学习既可以解释控制任务，也可以解释现代大语言模型后训练中的在线策略优化。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)

boxes = [
    (1.8, 2.5, 2.1, 1.2, "#4C78A8", "Agent\npolicy / value"),
    (6.0, 2.5, 2.4, 1.3, "#F58518", "Environment\nP(s', r | s, a)"),
    (10.2, 3.6, 1.8, 0.9, "#54A24B", "reward r_t"),
    (10.2, 1.4, 1.8, 0.9, "#E45756", "next state s_{t+1}"),
]
for x, y, w, h, color, text in boxes:
    rect = plt.Rectangle((x - w / 2, y - h / 2), w, h, facecolor=color, edgecolor="black")
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=11)

ax.annotate("", xy=(4.8, 2.5), xytext=(2.85, 2.5), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(3.9, 2.82, "action a_t", ha="center", fontsize=11)
ax.annotate("", xy=(2.85, 3.05), xytext=(8.0, 3.6), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(5.6, 3.9, "reward feedback", ha="center", fontsize=11)
ax.annotate("", xy=(2.85, 1.95), xytext=(8.0, 1.4), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(5.6, 0.95, "state transition", ha="center", fontsize=11)
ax.set_title("Agent-Environment Interaction in Reinforcement Learning")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)

states = [(2.0, 2.5, "state s"), (6.0, 2.5, "next state s'"), (10.0, 2.5, "bootstrap target")]
for x, y, text in states:
    ax.add_patch(plt.Circle((x, y), 0.6, color="#4C78A8"))
    ax.text(x, y, text, ha="center", va="center", color="white", fontsize=11)

ax.annotate("", xy=(5.35, 2.5), xytext=(2.65, 2.5), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(4.0, 2.85, "take action a", ha="center", fontsize=11)
ax.annotate("", xy=(9.35, 2.5), xytext=(6.65, 2.5), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(8.0, 2.95, "r + gamma * V(s')", ha="center", fontsize=11)
ax.text(6.0, 1.1, "Bellman update: current estimate is corrected by a one-step target", ha="center", fontsize=12)
ax.set_title("Bellman Backup")
plt.show()

## 核心公式与概念

1. 折扣回报
   $$
   G_t = \sum_{k=0}^{\infty}\gamma^k r_{t+k}
   $$

2. Bellman 期望方程
   $$
   V^\pi(s) = \sum_a \pi(a\mid s)\sum_{s',r} P(s',r\mid s,a)\left[r + \gamma V^\pi(s')\right]
   $$

3. Bellman 最优方程
   $$
   V^\star(s) = \max_a \sum_{s',r} P(s',r\mid s,a)\left[r + \gamma V^\star(s')\right]
   $$
   $$
   Q^\star(s,a) = \sum_{s',r} P(s',r\mid s,a)\left[r + \gamma \max_{a'} Q^\star(s',a')\right]
   $$

4. 优势函数
   $$
   A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)
   $$
   它描述“在给定状态下，该动作相对平均水平到底好多少”，是现代策略梯度算法中最常见的学习信号。

## 方法谱系

从知识结构上，可以把强化学习基础分成三层：

- **问题设置层**：MDP、回报、策略、价值函数、Bellman 方程；
- **估计层**：DP、MC、TD 分别对应模型已知、完整采样和一步 bootstrap；
- **控制层**：在学会评估后，通过贪心改进、策略梯度或策略迭代获得更优策略。

DQN 是把 TD 控制推广到深度函数逼近；Actor-Critic 是把价值估计与策略优化拼接成统一框架；PPO、SAC、GRPO 等方法都是在更复杂场景下继续改进“如何构造稳定的更新信号”。

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]



print("临时目录:", temp_root)

In [ ]:
states = ["S0", "S1", "S2", "Terminal"]
gamma = 0.9
policy = {
    0: {"stay": 0.25, "right": 0.75},
    1: {"stay": 0.15, "right": 0.85},
    2: {"stay": 0.10, "right": 0.90},
}
transitions = {
    0: {"stay": [(1.0, 0, 0.0)], "right": [(1.0, 1, 0.0)]},
    1: {"stay": [(1.0, 1, 0.0)], "right": [(1.0, 2, 0.0)]},
    2: {"stay": [(1.0, 2, 0.2)], "right": [(1.0, 3, 1.0)]},
}

def policy_evaluation(policy, transitions, gamma=0.9, iterations=20):
    V = np.zeros(4)
    history = [V.copy()]
    for _ in range(iterations):
        new_V = V.copy()
        for s in [0, 1, 2]:
            total = 0.0
            for action, action_prob in policy[s].items():
                for prob, next_state, reward in transitions[s][action]:
                    total += action_prob * prob * (reward + gamma * V[next_state])
            new_V[s] = total
        V = new_V
        history.append(V.copy())
    return np.array(history)

value_history = policy_evaluation(policy, transitions, gamma=gamma, iterations=25)
value_df = pd.DataFrame(value_history[:, :3], columns=["V(S0)", "V(S1)", "V(S2)"])
value_df.head()

In [ ]:
value_long = value_df.reset_index().melt(id_vars="index", var_name="state", value_name="value")
plt.figure(figsize=(10, 5))
sns.lineplot(data=value_long, x="index", y="value", hue="state", marker="o")
plt.title("Iterative Policy Evaluation Convergence")
plt.xlabel("iteration")
plt.ylabel("state value")
plt.show()

In [ ]:
true_values = np.arange(1, 6) / 6.0

def generate_random_walk_episode():
    state = 2
    episode = []
    while True:
        action = np.random.choice([-1, 1])
        next_state = state + action
        if next_state == -1:
            reward = 0.0
            episode.append((state, reward, None))
            return episode
        if next_state == 5:
            reward = 1.0
            episode.append((state, reward, None))
            return episode
        reward = 0.0
        episode.append((state, reward, next_state))
        state = next_state

def mc_prediction(alpha=0.05, episodes=200):
    values = np.full(5, 0.5)
    errors = []
    for _ in range(episodes):
        episode = generate_random_walk_episode()
        G = 0.0
        visited = set()
        for state, reward, _ in reversed(episode):
            G = reward + G
            if state not in visited:
                values[state] += alpha * (G - values[state])
                visited.add(state)
        errors.append(np.sqrt(np.mean((values - true_values) ** 2)))
    return np.array(errors)

def td_prediction(alpha=0.1, episodes=200):
    values = np.full(5, 0.5)
    errors = []
    for _ in range(episodes):
        episode = generate_random_walk_episode()
        for state, reward, next_state in episode:
            target = reward if next_state is None else reward + values[next_state]
            values[state] += alpha * (target - values[state])
        errors.append(np.sqrt(np.mean((values - true_values) ** 2)))
    return np.array(errors)

mc_errors = mc_prediction()
td_errors = td_prediction()
error_df = pd.DataFrame(
    {
        "episode": np.arange(1, len(mc_errors) + 1),
        "Monte Carlo": mc_errors,
        "TD(0)": td_errors,
    }
)
error_df.head()

In [ ]:
error_long = error_df.melt(id_vars="episode", var_name="method", value_name="rmse")
plt.figure(figsize=(10, 5))
sns.lineplot(data=error_long, x="episode", y="rmse", hue="method")
plt.title("Monte Carlo vs TD(0) on a Random Walk")
plt.ylabel("RMSE against true value")
plt.show()

## 建模与工程建议

- 如果环境模型已知，先从 DP 出发，因为它最能帮助理解 Bellman 结构。
- 如果采样代价昂贵，要优先考虑 TD 和 bootstrap，因为完整蒙特卡洛轨迹的方差通常过大。
- 强化学习实验需要显式报告随机种子、平均回报曲线与方差，不应只给出单次最优结果。
- 在进入深度强化学习之前，必须先把价值函数、优势函数、on-policy / off-policy、exploration / exploitation 这些概念彻底打通，否则后续算法只会变成公式堆叠。

## 主要资料

- Sutton, Barto. *Reinforcement Learning: An Introduction*（第二版）
- David Silver 的 RL 课程讲义
- Bellman 方程是整个强化学习理论的共同语言，后续所有 notebook 都会回到这里